# RAS tutorial — backfat thickness in Duroc pigs

A step-by-step, reproducible walk-through of the **RAS** (Regional Association
Score) method on a real livestock GWAS dataset. We inspect the raw data, then run
RAS **two ways** — the one-call wrapper and the step-by-step pipeline — on
chromosome 12 as a worked example.

> ⚠️ **Results withheld pending publication.** This public version is a *how-to*:
> it shows the data, the PLINK pre-processing, and how to call RAS. The detected
> region, result figures, and biological interpretation are omitted because they
> are part of a manuscript in preparation. Run the notebook yourself on the
> Figshare data to reproduce the full output.

**Dataset:** 352 Duroc pigs, 36,120 SNPs (Illumina PorcineSNP60), 18 autosomes.
Source: Figshare [10.6084/m9.figshare.4263317](https://doi.org/10.6084/m9.figshare.4263317).
Download the four data files into a `data/` folder next to this notebook before
running (see `data/README.md`).

**Trait:** backfat thickness at the 3/4 rib (`BFT34R`) — a classic, highly
polygenic carcass trait.

The whole notebook runs in a few minutes on a laptop.

## 0. Setup

We need `readxl` (to read the phenotype spreadsheet) and `RAS` itself. Step 4
also calls **PLINK 1.9** for the marginal GWAS — install it from
<https://www.cog-genomics.org/plink/> and make sure `plink` is on your PATH (or
edit the `plink` variable in Step 4).

```r
install.packages("readxl")
remotes::install_github("hepingzhangyale/RAS")
```

RAS averages over random 50/50 train/holdout splits, so we `set.seed()` for
reproducibility.

In [ ]:
library(readxl)
library(RAS)

set.seed(1)
data_dir <- "data" 

## 1. A tiny base-R reader for PLINK `.bed`

PLINK stores genotypes in a compact binary (SNP-major) format. The reader below
is ~20 lines of base R — no genetics package required. It returns an `n x m`
dosage matrix (count of allele A1: 0, 1, 2, or NA). The 2-bit encoding maps as
`00 = 2`, `01 = NA`, `10 = 1`, `11 = 0`.

In [ ]:
read_plink_bed <- function(bed, n, m) {
  bytes_per_snp <- ceiling(n / 4)
  con <- file(bed, "rb"); on.exit(close(con))
  magic <- readBin(con, "raw", 3)
  stopifnot(magic[1] == as.raw(0x6c),   # PLINK magic bytes
            magic[2] == as.raw(0x1b),
            magic[3] == as.raw(0x01))    # 0x01 = SNP-major
  raw_all <- readBin(con, "raw", bytes_per_snp * m)
  dim(raw_all) <- c(bytes_per_snp, m)
  lut  <- c(2L, NA_integer_, 1L, 0L)     # 2-bit code -> dosage
  geno <- matrix(NA_integer_, n, m)
  for (j in seq_len(m)) {
    b <- as.integer(raw_all[, j])
    g <- integer(bytes_per_snp * 4)
    g[seq(1, by = 4, length.out = bytes_per_snp)] <- lut[(b %%  4) + 1]
    g[seq(2, by = 4, length.out = bytes_per_snp)] <- lut[((b %/% 4)  %% 4) + 1]
    g[seq(3, by = 4, length.out = bytes_per_snp)] <- lut[((b %/% 16) %% 4) + 1]
    g[seq(4, by = 4, length.out = bytes_per_snp)] <- lut[((b %/% 64) %% 4) + 1]
    geno[, j] <- g[1:n]
  }
  geno
}

## 2. Load and inspect the data

### The sample file (`.fam`) and SNP map (`.bim`)

`.fam` has one row per animal; its phenotype column is a placeholder (`-9`).
`.bim` has one row per SNP: chromosome, SNP name, genetic distance, base-pair
position, and the two alleles.

In [ ]:
fam <- read.table(file.path(data_dir, "GWAS_genotypes.fam"),
                  header = FALSE, fill = TRUE)[, 1:6]
colnames(fam) <- c("FID", "IID", "PID", "MID", "SEX", "PHENO")
bim <- read.table(file.path(data_dir, "GWAS_genotypes.bim"), header = FALSE)
colnames(bim) <- c("CHR", "SNP", "CM", "BP", "A1", "A2")

head(fam, 5)
head(bim, 5)

  FID  IID PID MID SEX PHENO
1 248 1400   0   0   0    -9
2 249 1402   0   0   0    -9
3 297 1403   0   0   0    -9
4 121 1404   0   0   0    -9
5  96 1405   0   0   0    -9
  CHR         SNP CM     BP A1 A2
1   1 CASI0009658  0 145129  A  G
2   1 MARC0044150  0 286933  G  A
3   1 ASGA0000014  0 342481  A  C
4   1 H3GA0000026  0 389876  A  G
5   1 ASGA0000021  0 489855  A  C


### The genotype dosage matrix

Decode the `.bed` into a 352 × 36,120 matrix and mean-impute the 0.13 % missing
calls. Each entry is 0/1/2 (copies of allele A1); a corner and the value counts
are shown below.

In [ ]:
n <- nrow(fam); m <- nrow(bim)
geno <- read_plink_bed(file.path(data_dir, "GWAS_genotypes.bed"), n, m)

col_mean <- colMeans(geno, na.rm = TRUE)
na_idx   <- which(is.na(geno), arr.ind = TRUE)          # value counts below are pre-impute
cat("dim(geno):", nrow(geno), "x", ncol(geno), "\n")
table(factor(geno, levels = c(0, 1, 2)), useNA = "ifany")
if (nrow(na_idx)) geno[na_idx] <- col_mean[na_idx[, 2]]
geno[1:6, 1:8]

dim(geno): 352 x 36120

      0       1       2    <NA>
6810395 4779239 1108135   16471
     [,1]     [,2] [,3] [,4] [,5] [,6] [,7] [,8]
[1,]    0 0.000000    1    0    0    0    0    1
[2,]    0 0.000000    1    0    0    0    0    1
[3,]    1 1.000000    1    1    1    0    0    0
[4,]    0 1.000000    0    1    1    1    1    1
[5,]    0 0.000000    1    0    0    0    0    1
[6,]    0 0.566092    0    1    1    1    1    1


### The phenotypes

The real phenotypes are in the Excel file. We align them to the genotype sample
order by family + individual ID, then take `BFT34R`. Note all animals are male.

In [ ]:
ph <- as.data.frame(read_excel(
  file.path(data_dir, "gwas_duroc_phenotypes.xlsx"), sheet = 1))[, 1:9]
colnames(ph)[1:2] <- c("FID", "IID")
ph <- ph[match(paste(fam$FID, fam$IID), paste(ph$FID, ph$IID)), ]

head(ph[, c("FID","IID","sex","LiveWeight","CW","BFT34R","BFTLR","HW")], 5)

trait <- "BFT34R"
y     <- suppressWarnings(as.numeric(ph[[trait]]))
summary(y)

  FID  IID  sex LiveWeight    CW BFT34R BFTLR    HW
1 248 1400 male        135 102.5     29    29  12.2
2 249 1402 male        108  81.5     20    18 10.42
3 297 1403 male         86  58.0     12    11  7.78
4 121 1404 male        113  83.0     24    22 10.54
5  96 1405 male        116  88.5     27    24 11.32
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.
  12.00   29.00   33.00   33.53   37.00   59.00


## 3. Covariates: genotype principal components

RAS needs at least one covariate. Since sex is constant (all male), we use the
top three genotype PCs — the standard way to control for population structure and
relatedness.

In [ ]:
af    <- colMeans(geno) / 2
maf   <- pmin(af, 1 - af)
pcs   <- prcomp(scale(geno[, maf > 0.05]), center = FALSE, scale. = FALSE)
covdf <- data.frame(pc1 = pcs$x[, 1], pc2 = pcs$x[, 2], pc3 = pcs$x[, 3])
cat(sprintf("PC1-3 explain %.1f%% of genotypic variance\n",
            100 * sum(pcs$sdev[1:3]^2) / sum(pcs$sdev^2)))
head(round(covdf, 3), 5)

PC1-3 explain 15.9% of genotypic variance
      pc1    pc2     pc3
1  -2.916  2.988  18.185
2   3.498 -6.412  10.822
3  -7.195 -1.796   3.492
4 -34.613 -5.515  32.444
5 -28.270 -8.218 -24.556


## 4. Context: a marginal single-SNP GWAS with PLINK on chromosome 12

Before RAS, run the ordinary per-SNP scan — here with **PLINK 1.9**
(`--linear`, additive quantitative model). We write a PLINK phenotype file, run
the association on chromosome 12, and plot the results. This is exactly the
situation where RAS helps: regional signal that no individual test can establish
on its own.

> *Result figure withheld — run the cell to reproduce it.*

In [ ]:
sel    <- which(bim$CHR == 12)
geno12 <- geno[, sel]
bim12  <- bim[sel, ]

# PLINK phenotype file: FID IID BFT34R  (missing coded as -9)
pheno_tab <- data.frame(FID = fam$FID, IID = fam$IID,
                        BFT34R = ifelse(is.na(y), -9, y))
write.table(pheno_tab, "results/pheno_BFT34R.txt", row.names = FALSE, quote = FALSE)

plink <- "plink"          # PLINK 1.9 executable (on PATH, or a full path)
system2(plink, c("--bfile", "data/GWAS_genotypes", "--chr", "12",
                 "--pheno", "results/pheno_BFT34R.txt", "--pheno-name", "BFT34R",
                 "--linear", "--allow-no-sex", "--out", "results/chr12_plink"))

gw <- read.table("results/chr12_plink.assoc.linear", header = TRUE)
gw <- gw[gw$TEST == "ADD" & is.finite(gw$P), ]
cat(sprintf("PLINK top SNP: %s at %.2f Mb  (-log10p = %.2f)\n",
            gw$SNP[which.max(-log10(gw$P))],
            gw$BP[which.max(-log10(gw$P))] / 1e6, max(-log10(gw$P))))

bonf <- -log10(0.05 / m)
plot(gw$BP / 1e6, -log10(gw$P), pch = 19, cex = 0.5, col = "#2c7bb6",
     ylim = c(0, bonf * 1.05),
     xlab = "Chr 12 position (Mb)", ylab = expression(-log[10](p)),
     main = "Marginal single-SNP GWAS (PLINK --linear)  -  backfat thickness (BFT34R)")
abline(h = bonf, col = "red", lty = 2)
text(2, bonf, "Bonferroni 0.05", col = "red", pos = 3, cex = 0.9)

## 5. How RAS works

RAS runs in four stages:

1. **Scan** (`ras_scan`) — for each of `num_rep` random 50/50 train/holdout
   splits: fit per-SNP GWAS weights on the training half, build a weighted-dosage
   (PGS) matrix for the holdout half, and slide an expanding window recording the
   minimum p-value at each grid position. The −log10(p) profiles are averaged.
2. **Detect** (`ras_detect`) — slide a window over the averaged profile and, at
   each position, fit a segmented model + Davies test; keep candidates that are a
   significant peak (rising on the left, falling on the right).
3. **Validate** (`ras_validate`) — re-test each candidate locally and drop
   low-signal ones (−log10p ≤ `min_signal`).
4. **Plot** — visualise the profile and the detected changepoints.

Below we run this **two equivalent ways**.

## 6. Method A — the one-call wrapper `ras()`

`ras()` runs all four stages in a single call and returns an object of class
`"ras"`. This is the recommended entry point.

In [ ]:
set.seed(1)
resA <- ras(
  geno           = geno12,
  phenotype      = y,
  covariates     = covdf,
  covariate_cols = c("pc1", "pc2", "pc3"),
  is_continuous  = TRUE,
  num_rep        = 10,      # average over 10 splits
  skip1          = 5,       # profile-grid stride (SNP units)
  skip2          = 10,
  chrom          = 12,
  save_dir       = "results",
  run_plots      = FALSE
)
print(resA)

Map the detected index back to a physical position:

In [ ]:
tau <- resA$detection$tau_hats
cat(sprintf("changepoint: index %d  ->  %s at %.2f Mb\n",
            tau, bim12$SNP[tau], bim12$BP[tau] / 1e6))

## 7. Method B — the step-by-step pipeline

The same analysis, run stage by stage. This gives you the intermediate objects
(useful for tuning detection parameters without re-running the expensive scan).

### 7a. `ras_scan()` — the averaged regional profile

Returns a list with `x` (the SNP-index grid) and `y` (the averaged −log10(p)
profile). This is the expensive stage (it runs the `num_rep` splits).

In [ ]:
set.seed(1)
scan <- ras_scan(
  geno           = geno12,
  phenotype      = y,
  covariates     = covdf,
  covariate_cols = c("pc1", "pc2", "pc3"),
  is_continuous  = TRUE,
  num_rep        = 10, skip1 = 5, skip2 = 10,
  chrom = 12, save_dir = "results"
)

cat("scan has elements:", paste(names(scan), collapse = ", "), "\n")
cat("profile length:", length(scan$y), "\n")
cat("peak -log10(p):", round(max(scan$y), 3),
    "at SNP index", scan$x[which.max(scan$y)], "\n")

plot(scan$x, scan$y, type = "l", col = "#2c7bb6", lwd = 1.5,
     xlab = "SNP index", ylab = expression(-log[10](p)),
     main = "ras_scan() output: averaged regional -log10(p) profile (chr 12)")
abline(h = 2.5, lty = 3, col = "grey50")

### 7b. `ras_detect()` — first-pass changepoint detection

Slides a window over the profile, fitting a segmented model + Davies test at each
position to find peak-shaped changepoints. Defaults match what `ras()` uses
internally.

In [ ]:
cp_result <- ras_detect(
  x = scan$x, y = scan$y,
  p.values.threshold             = 0.01,
  min.length                     = 10,
  window_size                    = 3000,
  slope_check_window_size        = 30,
  slope.p.values.threshold.left  = 1e-10,
  slope.p.values.threshold.right = 1e-20
)
cat("cp_result elements:", paste(names(cp_result), collapse = ", "), "\n")

### 7c. `ras_validate()` — second-pass validation

Re-tests each first-pass candidate locally and discards low-signal ones. Returns
the final validated changepoints in genomic (SNP) coordinates. `this.skip` must
match the scan's `skip1`.

In [ ]:
detection <- ras_validate(
  this.result        = cp_result,
  x = scan$x, y = scan$y,
  this.start         = 1,
  this.skip          = 5,          # = skip1
  second_window_size = 50,
  p.value.threshold  = 1e-10,
  min_signal         = 2.5
)
tau <- detection$tau_hats
cat(sprintf("validated changepoint: index %d -> %s at %.2f Mb\n",
            tau, bim12$SNP[tau], bim12$BP[tau] / 1e6))

### 7d. Both methods agree

Assemble the step-by-step results into a `"ras"` object (exactly what `ras()`
builds internally) and confirm it matches Method A.

In [ ]:
resB <- structure(
  list(scan = scan, detection = detection, chrom = 12, save_dir = "results"),
  class = "ras"
)

cat("Method A changepoint:", resA$detection$tau_hats, "\n")
cat("Method B changepoint:", resB$detection$tau_hats, "\n")
cat("identical:", identical(resA$detection$tau_hats, resB$detection$tau_hats), "\n")

## 8. Visualise the result

`plot()` on a `"ras"` object draws the scan profile with the detected changepoint
(red dashed line; green/purple = fitted left/right slopes) and a dual-axis overlay
of the changepoint-test significance. Use `device = "screen"` to render inline.

In [ ]:
plot(resA, device = "screen")

Zoomed view of the detected region:

In [ ]:
plot(resA, zoom = TRUE, device = "screen")

## 9. Takeaways

> **Results withheld pending publication.** The specific changepoint location,
> significance values, and biological interpretation produced by this notebook
> are part of a manuscript in preparation and are omitted here. Running the
> notebook on the Figshare data reproduces the full result.

What this tutorial shows even with the results hidden:

- How to go from raw PLINK files to a genotype dosage matrix, aligned phenotypes,
  and PC covariates.
- How to run a marginal single-SNP GWAS with PLINK for context.
- How to run RAS **two equivalent ways** — the one-call `ras()` and the
  step-by-step `ras_scan → ras_detect → ras_validate` — and how to map a detected
  changepoint index back to a physical SNP position.

> **Note on genomic positions:** base-pair coordinates in this dataset follow the
> older **Sscrofa 10.2 assembly**, so the same SNP can sit elsewhere in newer
> assemblies. **Match loci by SNP name, not raw position.**

### Try your own variations
- **Other traits:** `trait <- "CW"` (carcass weight), `"LE%"`, `"LiveWeight"`.
- **Other chromosomes:** change `bim$CHR == 12`, or loop over `1:18`.
- **Binary traits:** `is_continuous = FALSE` with `scan_test = "score"` for the
  fast Rao score test.
- **Resolution vs. speed:** tune `skip1`/`skip2` and `num_rep`.

### References
> **Method:** Y. Jiang & H. Zhang. *Empowering genome-wide association studies via
> a visualizable test based on the regional association score.* PNAS 122(9)
> e2419721122 (2025). https://doi.org/10.1073/pnas.2419721122
>
> **Dataset (source):** P. G. Eusebi et al. *A genome-wide association analysis for
> carcass traits in a commercial Duroc pig population.* Animal Genetics 48:466–469
> (2017). https://doi.org/10.1111/age.12545